In [2]:
import numpy as np, zarr, json, torch, glob, re, pandas as pd
from pathlib import Path
from scipy import ndimage as ndi

VERSION = 'v3'
OVERWRITE = True
TEST_ZARR = "../train_processed/6bba_1f58c2f6.zarr/0"
GEFF_PATH = "../train/6bba_1f58c2f6.geff"
MASK_PATH = "deadends/stardist/cellpose_mask/cellpose_cache/6bba_1f58c2f6_masks.npy"
MIN_VOX = 50   # drop tiny cellpose objects (debris); set from cell volume

In [3]:
g = zarr.open(GEFF_PATH)
gt = np.column_stack([g[f"nodes/props/{a}/values"][:] for a in ("t", "z", "y", "x")]).astype(float)
gt_t = gt[:, 0].astype(int)

frames = np.unique(gt_t)
gt_by_frame = {int(t): gt[gt_t == t, 1:] for t in frames}
print(len(gt), "GT points;", len(frames))

z = zarr.open(TEST_ZARR, mode="r")
vol = np.stack([np.asarray(z[int(t)]) for t in frames]).astype(np.float32)
print("vol", vol.shape)

# cellpose centroids per used frame
masks = np.load(MASK_PATH, mmap_mode="r")
print("masks", masks.shape, masks.dtype)
assert masks.ndim == 4 and masks.shape[1:] == vol.shape[1:], "mask/raw shape mismatch, check frame indexing"

cp_by_frame = {}
for t in frames:
    m = np.asarray(masks[int(t)])
    ids, counts = np.unique(m, return_counts=True)
    keep = (ids > 0) & (counts >= MIN_VOX)
    ids = ids[keep]
    cp_by_frame[int(t)] = (np.array(ndi.center_of_mass(np.ones(m.shape), m, ids)).reshape(-1, 3))

n_cp = sum(len(v) for v in cp_by_frame.values())
print("cellpose centroids:", n_cp, "vs GT:", len(gt[np.isin(gt_t, frames)]))

317 GT points; 99
vol (99, 64, 256, 256)
masks (100, 64, 256, 256) uint16
cellpose centroids: 10223 vs GT: 317


In [5]:
from unet3d import UNet3D
device = "cuda" if torch.cuda.is_available() else "cpu"
nrm = json.load(open(f"models/norm_{VERSION}.json"))
x_all = np.clip((vol - nrm["lo"]) / (nrm["hi"] - nrm["lo"] + 1e-8), 0, 1).astype(np.float32)

Path("preds").mkdir(exist_ok=True)
ckpts = sorted(glob.glob(f"models/unet_{VERSION}_e*.pt"), key=lambda p: int(re.findall(r"e(\d+)\.pt", p)[0]))
model = UNet3D(16).to(device).eval()

for ck in ckpts:
    ep = int(re.findall(r"e(\d+)\.pt", ck)[0])
    out = Path(f"preds/e_{VERSION}_{ep}.npy")
    
    if not OVERWRITE and out.exists():
        print("using existing:", out)
        continue

    model.load_state_dict(torch.load(ck, map_location=device))
    preds = []
    with torch.no_grad():
        for f in x_all:
            xb = torch.from_numpy(f)[None, None].to(device)
            with torch.autocast(device_type=device, enabled=(device == "cuda")):
                p = model(xb)
            preds.append(p[0, 0].float().cpu().numpy().astype(np.float16))
    
    np.save(out, np.stack(preds))
    print("saved: ", out)

saved:  preds/e_v3_10.npy
saved:  preds/e_v3_20.npy
saved:  preds/e_v3_30.npy
saved:  preds/e_v3_40.npy
saved:  preds/e_v3_50.npy
saved:  preds/e_v3_60.npy
saved:  preds/e_v3_70.npy
saved:  preds/e_v3_80.npy
saved:  preds/e_v3_90.npy
saved:  preds/e_v3_100.npy


In [ ]:
from skimage.feature import peak_local_max
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
DISTANCES = [4, 5, 6]      # xy half-window in pixels
PHYS = np.array([1.625, 0.40625, 0.40625])   # µm per voxel (z,y,x)
MATCH_RADIUS = 7.0

def n_matched(dets, ref):
    if len(dets) == 0 or len(ref) == 0:
        return 0
    D = cdist(dets * PHYS, ref * PHYS)
    r, c = linear_sum_assignment(D)
    return int((D[r, c] <= MATCH_RADIUS).sum())

# reference per frame = GT  U  cellpose centroids not already near a GT point
ref_by_frame = {}
for t in frames:
    t = int(t); G, C = gt_by_frame[t], cp_by_frame[t]
    if len(G) and len(C):
        C = C[cdist(C * PHYS, G * PHYS).min(1) > MATCH_RADIUS]
    ref_by_frame[t] = np.vstack([G, C]) if len(C) else G

n_gt = sum(len(v) for v in gt_by_frame.values())
n_ref = sum(len(v) for v in ref_by_frame.values())

rows = []
for ck in ckpts:
    ep = int(re.findall(r"e(\d+)\.pt", ck)[0])
    P = np.load(f"preds/e_{VERSION}_{ep}.npy").astype(np.float32)
    for d in DISTANCES:
        dz = max(1, round(d / 4))
        fp = np.ones((2*dz + 1, 2*d + 1, 2*d + 1), bool)
        peaks = []   # computed once at the lowest threshold, then filtered
        for f in P:
            c = peak_local_max(f, footprint=fp, threshold_abs=min(THRESHOLDS), exclude_border=False)
            peaks.append((c, f[tuple(c.T)] if len(c) else np.array([])))
        for t in THRESHOLDS:
            hit_gt = hit_ref = ndet = 0
            for (c, v), fr in zip(peaks, frames):
                dets = c[v >= t]; ndet += len(dets)
                hit_gt += n_matched(dets, gt_by_frame[int(fr)])
                hit_ref += n_matched(dets, ref_by_frame[int(fr)])
            prec = hit_ref / max(ndet, 1)
            rec_ref = hit_ref / n_ref
            rows.append(dict(epoch=ep, dist=d, thr=t,
                             recall_gt=hit_gt / n_gt,
                             agree_prec=prec, agree_rec=rec_ref,
                             agree_f1=2*prec*rec_ref / max(prec + rec_ref, 1e-9),
                             n_det=ndet))

res = pd.DataFrame(rows)
res.to_csv(f"sweeps/unet_sweep_{VERSION}.csv", index=False)
top = res.sort_values("agree_f1", ascending=False)
print(top.head(10).to_string(index=False))
print("\nBEST:", dict(top.iloc[0]))

 epoch  dist  thr  recall_gt  agree_prec  agree_rec  agree_f1  n_det
    10     6  0.8   0.798107    0.324904   0.740823  0.451704  23727
    10     4  0.8   0.798107    0.324467   0.742072  0.451513  23799
    10     5  0.8   0.798107    0.324437   0.741784  0.451430  23792
    30     6  0.8   0.854890    0.299671   0.769940  0.431425  26736
    30     5  0.8   0.854890    0.299481   0.770229  0.431273  26763
    30     4  0.8   0.854890    0.299313   0.770229  0.431099  26778
    20     6  0.8   0.842271    0.296662   0.780511  0.429917  27378
    20     5  0.8   0.842271    0.296402   0.781280  0.429761  27429
    20     4  0.8   0.842271    0.296317   0.781568  0.429715  27447
    10     6  0.7   0.883281    0.279728   0.855180  0.421564  31813

BEST: {'epoch': np.float64(10.0), 'dist': np.float64(6.0), 'thr': np.float64(0.8), 'recall_gt': np.float64(0.7981072555205048), 'agree_prec': np.float64(0.3249041176718506), 'agree_rec': np.float64(0.7408226023448011), 'agree_f1': np.float6

: 

mdoel v2 stats

| epoch | dist | thr | recall\_gt | agree\_prec | agree\_rec | agree\_f1 | n\_det |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 10 | 6 | 0.6 | 0.960222 | 0.654970 | 0.876302 | 0.749640 | 8350 |
| 60 | 6 | 0.9 | 0.916744 | 0.667349 | 0.836404 | 0.742374 | 7822 |
| 40 | 6 | 0.7 | 0.937095 | 0.655954 | 0.853549 | 0.741819 | 8121 |
| 10 | 5 | 0.6 | 0.965772 | 0.637296 | 0.886557 | 0.741540 | 8682 |
| 60 | 5 | 0.9 | 0.920444 | 0.663706 | 0.838007 | 0.740741 | 7880 |
| 60 | 4 | 0.9 | 0.920444 | 0.662491 | 0.838808 | 0.740296 | 7902 |
| 40 | 5 | 0.7 | 0.937095 | 0.648076 | 0.858356 | 0.738540 | 8266 |
| 40 | 4 | 0.7 | 0.938020 | 0.645359 | 0.862202 | 0.738185 | 8338 |
| 10 | 4 | 0.6 | 0.966698 | 0.629567 | 0.891844 | 0.738098 | 8841 |
| 40 | 6 | 0.6 | 0.943571 | 0.642187 | 0.865887 | 0.737445 | 8415 |

model v1 stats

| epoch | dist | thr | recall\_gt | agree\_prec | agree\_rec | agree\_f1 | n\_det |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 40 | 6 | 0.8 | 0.925994 | 0.663897 | 0.870694 | 0.753362 | 8185 |
| 50 | 6 | 0.6 | 0.926920 | 0.662126 | 0.872296 | 0.752818 | 8222 |
| 90 | 6 | 0.7 | 0.917669 | 0.669455 | 0.859317 | 0.752596 | 8011 |
| 50 | 6 | 0.5 | 0.933395 | 0.654115 | 0.885115 | 0.752281 | 8445 |
| 90 | 6 | 0.6 | 0.925994 | 0.662073 | 0.869893 | 0.751887 | 8200 |
| 50 | 6 | 0.4 | 0.938945 | 0.646916 | 0.893927 | 0.750622 | 8624 |
| 50 | 6 | 0.7 | 0.914894 | 0.670152 | 0.852267 | 0.750317 | 7937 |
| 30 | 6 | 0.8 | 0.921369 | 0.658478 | 0.871815 | 0.750276 | 8263 |
| 40 | 5 | 0.8 | 0.925994 | 0.656671 | 0.873738 | 0.749811 | 8304 |
| 40 | 4 | 0.8 | 0.925994 | 0.656280 | 0.874059 | 0.749674 | 8312 |



model v0 stats


| epoch | dist | thr | recall\_gt | agree\_prec | agree\_rec | agree\_f1 | n\_det |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 50 | 6 | 0.2 | 0.886216 | 0.637027 | 0.782887 | 0.702466 | 7670 |
| 50 | 6 | 0.3 | 0.851989 | 0.651127 | 0.754206 | 0.698886 | 7229 |
| 50 | 5 | 0.2 | 0.886216 | 0.628861 | 0.786252 | 0.698804 | 7803 |
| 50 | 6 | 0.1 | 0.920444 | 0.602681 | 0.828553 | 0.697794 | 8580 |
| 50 | 5 | 0.3 | 0.851989 | 0.645907 | 0.756129 | 0.696686 | 7306 |
| 50 | 4 | 0.2 | 0.887142 | 0.623952 | 0.787214 | 0.696139 | 7874 |
| 50 | 4 | 0.3 | 0.851989 | 0.642692 | 0.755969 | 0.694743 | 7341 |
| 50 | 6 | 0.4 | 0.828862 | 0.655823 | 0.730011 | 0.690931 | 6947 |
| 50 | 5 | 0.4 | 0.828862 | 0.652429 | 0.731774 | 0.689827 | 7000 |
| 50 | 4 | 0.4 | 0.828862 | 0.651468 | 0.732575 | 0.689645 | 7018 |

old model v0 stats


| epoch | dist | thr | recall\_gt | agree\_prec | agree\_rec | agree\_f1 | n\_det |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 60 | 6 | 0.6 | 0.851989 | 0.664122 | 0.794584 | 0.723519 | 7467 |
| 60 | 5 | 0.6 | 0.851989 | 0.659891 | 0.795866 | 0.721528 | 7527 |
| 60 | 6 | 0.7 | 0.841813 | 0.669689 | 0.780965 | 0.721059 | 7278 |
| 60 | 4 | 0.6 | 0.851989 | 0.656981 | 0.796988 | 0.720243 | 7571 |
| 60 | 5 | 0.7 | 0.841813 | 0.666758 | 0.781926 | 0.719764 | 7319 |
| 60 | 4 | 0.7 | 0.841813 | 0.665396 | 0.782887 | 0.719376 | 7343 |
| 80 | 6 | 0.6 | 0.876041 | 0.668134 | 0.778401 | 0.719065 | 7271 |
| 60 | 6 | 0.8 | 0.806660 | 0.679712 | 0.756930 | 0.716246 | 6950 |
| 80 | 4 | 0.6 | 0.876041 | 0.661597 | 0.780644 | 0.716207 | 7364 |
| 60 | 5 | 0.8 | 0.806660 | 0.678469 | 0.758372 | 0.716199 | 6976 |